# 03. 심화: Positional Encoding과 Self-Attention 직관

목표: NeuralLog의 Transformer 분류기가 왜 로그 시퀀스 문맥을 볼 수 있는지, 작은 self-attention 계산으로 이해합니다.

실행 방법: 모든 셀을 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. Positional encoding 구현

NeuralLog 구현은 sinusoidal positional encoding을 입력 벡터에 더합니다. 같은 로그 메시지라도 시퀀스 내 위치가 다르면 모델이 구분할 수 있습니다.

In [ ]:
import hashlib
import math
import re


def positional_encoding(position, dim):
    values = []
    for i in range(dim):
        angle = position / (10000 ** ((2 * (i // 2)) / dim))
        values.append(math.sin(angle) if i % 2 == 0 else math.cos(angle))
    return values


for pos in range(3):
    print(pos, [round(value, 3) for value in positional_encoding(pos, 6)])

## 2. 메시지 벡터와 위치 정보 결합

BERT 임베딩 대신 해시 기반 toy embedding을 사용합니다. 위치 벡터를 더해 같은 메시지도 위치에 따라 조금 다른 표현을 갖게 합니다.

In [ ]:
def token_vector(token, dim=8):
    digest = hashlib.sha256(token.encode("utf-8")).digest()
    return [(digest[i] / 127.5) - 1.0 for i in range(dim)]


def mean_vector(vectors, dim=8):
    if not vectors:
        return [0.0] * dim
    return [sum(vector[i] for vector in vectors) / len(vectors) for i in range(dim)]


def message_vector(message, dim=8):
    words = re.findall(r"[a-z]+", message.lower())
    return mean_vector([token_vector(word, dim) for word in words], dim)


def add_vectors(a, b):
    return [x + y for x, y in zip(a, b)]


sequence = [
    "disk read completed",
    "network link recovered",
    "disk read timeout",
    "kernel reports failure",
]

encoded = [add_vectors(message_vector(message), positional_encoding(i, 8)) for i, message in enumerate(sequence)]
print("encoded length:", len(encoded), "vector dim:", len(encoded[0]))

## 3. Single-head self-attention

실제 Transformer는 Q/K/V projection과 multi-head attention을 사용합니다. 여기서는 입력 벡터를 그대로 Q/K/V로 두고 attention weight만 계산합니다.

In [ ]:
def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def softmax(values):
    m = max(values)
    exps = [math.exp(value - m) for value in values]
    total = sum(exps)
    return [value / total for value in exps]


def attention_for_query(query_index, vectors):
    scale = math.sqrt(len(vectors[0]))
    scores = [dot(vectors[query_index], key) / scale for key in vectors]
    weights = softmax(scores)
    attended = []
    for dim_index in range(len(vectors[0])):
        attended.append(sum(weight * value[dim_index] for weight, value in zip(weights, vectors)))
    return weights, attended


for i, message in enumerate(sequence):
    weights, _attended = attention_for_query(i, encoded)
    pretty = ", ".join(f"{w:.2f}" for w in weights)
    print(f"query={i} {message:24s} weights=[{pretty}]")

## 4. Threshold sweep

운영 이상 탐지에서는 threshold 선택이 중요합니다. 같은 score라도 threshold에 따라 precision과 recall의 균형이 달라집니다.

In [ ]:
scored_windows = [
    (0.10, 0),
    (0.20, 0),
    (0.35, 0),
    (0.55, 1),
    (0.65, 0),
    (0.72, 1),
    (0.88, 1),
    (0.95, 1),
]


def metrics_at_threshold(threshold):
    preds = [int(score >= threshold) for score, _label in scored_windows]
    labels = [label for _score, label in scored_windows]
    tp = sum(1 for y, p in zip(labels, preds) if y == 1 and p == 1)
    fp = sum(1 for y, p in zip(labels, preds) if y == 0 and p == 1)
    fn = sum(1 for y, p in zip(labels, preds) if y == 1 and p == 0)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


print("threshold | precision | recall | f1")
print("--- | --- | --- | ---")
for threshold in [0.30, 0.50, 0.70, 0.90]:
    precision, recall, f1 = metrics_at_threshold(threshold)
    print(f"{threshold:.2f} | {precision:.2f} | {recall:.2f} | {f1:.2f}")